# Medical Images — Download + Sample

**Run once.** Download two Kaggle datasets, looks at how their folders are organized, and copies a small resized sample into `data/medical_small/`.

In [2]:
   %pip install -q pandas pillow matplotlib scikit-learn pyarrow kagglehub


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# !pip install -q kagglehub

import os

import pandas as pd
import PIL.Image as PImage


import kagglehub

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ct_path = kagglehub.dataset_download("shuvokumarbasakbd/brain-ct-medical-imaging-colorized-dataset")
tumor_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("CT:", ct_path)
print("Tumor:", tumor_path)

100%|██████████| 93.1M/93.1M [00:01<00:00, 59.9MB/s]

Extracting files...


100%|██████████| 157M/157M [00:01<00:00, 127MB/s]  

Extracting files...


CT: /home/codespace/.cache/kagglehub/datasets/shuvokumarbasakbd/brain-ct-medical-imaging-colorized-dataset/versions/1
Tumor: /home/codespace/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


In [5]:
# looking at the top folders
!ls {ct_path}
print("-----")
!ls {tumor_path}

'Computed Tomography (CT) of the Brain'
-----
Testing  Training


In [6]:
# listing images in both of the datasets

datasets = [("brain_ct_color", ct_path), ("brain_tumor_mri", tumor_path)]

rows = []

for dataset, base in datasets:
  for folder, subfolders, files in os.walk(base):
    for f in files:
      ext = f.lower().split(".")[-1]
      if ext in ["jpg", "jpeg", "png", "bmp", "tif", "tiff"]:
        rows.append({
          "dataset": dataset,
          "label": os.path.basename(folder),
          "split": os.path.basename(os.path.dirname(folder)),
          "file": f,
          "orig_path": os.path.join(folder, f),
        })

files_df = pd.DataFrame(rows)

print(files_df.shape)
files_df[["dataset", "split", "label"]].value_counts().sort_index()

(9934, 5)


dataset          split     label     
brain_ct_color   test      aneurysm        48
                           cancer          48
                           tumor           48
                 train     aneurysm       840
                           cancer         910
                           tumor          840
brain_tumor_mri  Testing   glioma         400
                           meningioma     400
                           notumor        400
                           pituitary      400
                 Training  glioma        1400
                           meningioma    1400
                           notumor       1400
                           pituitary     1400
Name: count, dtype: int64

In [7]:
# small sample for data prep
SAMPLE_PER_LABEL = 40

sample_parts = []

for dataset in files_df["dataset"].unique():
  ds_df = files_df[files_df["dataset"] == dataset]

  for label in ds_df["label"].unique():
    part = ds_df[ds_df["label"] == label]
    n = min(SAMPLE_PER_LABEL, len(part))
    sample_parts.append(part.sample(n=n, random_state=1010))

sample_df = pd.concat(sample_parts).reset_index(drop=True)

print(sample_df.shape)
sample_df[["dataset", "label"]].value_counts().sort_index()

(280, 5)


dataset          label     
brain_ct_color   aneurysm      40
                 cancer        40
                 tumor         40
brain_tumor_mri  glioma        40
                 meningioma    40
                 notumor       40
                 pituitary     40
Name: count, dtype: int64

In [9]:
# resizing + saving the sample
# - 128 x 128 pixels--> same number for each image-- stays consistent
SMALL = 128

for dataset in sample_df["dataset"].unique():
  for label in sample_df[sample_df["dataset"] == dataset]["label"].unique():
    !mkdir -p "data/medical_small/{dataset}/{label}"

small_paths = []
widths = []
heights = []
modes = []

for i in range(len(sample_df)):
  row = sample_df.iloc[i]
  mimg = PImage.open(row["orig_path"])

  widths.append(mimg.size[0])
  heights.append(mimg.size[1])
  modes.append(mimg.mode)

  # number in front of the name so files with the same name don't overwrite each other
  name = str(i) + "_" + row["file"].split(".")[0] + ".png"
  new_path = "data/medical_small/" + row["dataset"] + "/" + row["label"] + "/" + name

  mimg.convert("RGB").resize((SMALL, SMALL)).save(new_path)
  small_paths.append(new_path)

sample_df["small_path"] = small_paths
sample_df["orig_width"] = widths
sample_df["orig_height"] = heights
sample_df["orig_mode"] = modes

sample_df.to_csv("data/medical_index.csv", index=False)

display(sample_df.head())

,dataset,label,split,file,orig_path,small_path,orig_width,orig_height,orig_mode
0,brain_ct_color,aneurysm,train,71_4.jpg,/home/codespace/.cache/kagglehub/datasets/shuv...,data/medical_small/brain_ct_color/aneurysm/0_7...,512,512,RGB
1,brain_ct_color,aneurysm,train,57_3.jpg,/home/codespace/.cache/kagglehub/datasets/shuv...,data/medical_small/brain_ct_color/aneurysm/1_5...,512,512,RGB
2,brain_ct_color,aneurysm,train,24_2.jpg,/home/codespace/.cache/kagglehub/datasets/shuv...,data/medical_small/brain_ct_color/aneurysm/2_2...,512,512,RGB
3,brain_ct_color,aneurysm,train,43_8.jpg,/home/codespace/.cache/kagglehub/datasets/shuv...,data/medical_small/brain_ct_color/aneurysm/3_4...,512,512,RGB
4,brain_ct_color,aneurysm,train,21_2.jpg,/home/codespace/.cache/kagglehub/datasets/shuv...,data/medical_small/brain_ct_color/aneurysm/4_2...,512,512,RGB


In [10]:
# thinking about the glitch art componenet
# make a folder + empty file + catalogs elements of the work

!mkdir -p data/glitch

CREDITS = "data/glitch/glitch_credits.csv"

if os.path.exists(CREDITS):
  print("credits file already exists:")
  display(pd.read_csv(CREDITS))
else:
  credits_df = pd.DataFrame(columns=["file", "artist", "title", "year", "source_url", "method", "notes"])
  credits_df.to_csv(CREDITS, index=False)
  print("made empty", CREDITS)

made empty data/glitch/glitch_credits.csv
